In [2]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, log_loss

df = pd.read_csv("../data/processed/features_v2.csv", parse_dates=["GAME_DATE"])

old_feats = ["HOME_PTS_ROLL10", "HOME_PTS_ALLOWED_ROLL10", "HOME_WIN_ROLL10", "HOME_REST_DAYS",
             "AWAY_PTS_ROLL10", "AWAY_PTS_ALLOWED_ROLL10", "AWAY_WIN_ROLL10", "AWAY_REST_DAYS"]
xgb_feats = old_feats + ["ELO_DIFF", "NET5_DIFF", "NET10_DIFF", "NET20_DIFF", "REST_DIFF"]
target_col = "HOME_WIN"

assert "HOME_PTS" not in xgb_feats and "AWAY_PTS" not in xgb_feats
assert target_col not in xgb_feats

usable = df.dropna(subset=xgb_feats)

val_start = pd.Timestamp("2021-08-01")
test_start = pd.Timestamp("2022-08-01")
train = usable[usable["GAME_DATE"] < val_start]
val = usable[(usable["GAME_DATE"] >= val_start) & (usable["GAME_DATE"] < test_start)]
test = usable[usable["GAME_DATE"] >= test_start]

print("Train:", len(train), "| Val:", len(val), "| Test:", len(test))

Train: 6899 | Val: 1230 | Test: 3685


In [3]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

lr_feats = old_feats + ["ELO_DIFF"]

lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
lr.fit(train[lr_feats], train[target_col])
lr_val_probs = lr.predict_proba(val[lr_feats])[:, 1]

print("Logistic on validation:",
      f"acc {accuracy_score(val[target_col], (lr_val_probs >= 0.5).astype(int)):.4f}",
      f"log loss {log_loss(val[target_col], lr_val_probs):.4f}")

Logistic on validation: acc 0.6618 log loss 0.6333


In [4]:
from xgboost import XGBClassifier

def fit_xgb(max_depth, n_estimators=1000):
    model = XGBClassifier(
        n_estimators=n_estimators,
        learning_rate=0.02,
        max_depth=max_depth,
        min_child_weight=20,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=5,
        eval_metric="logloss",
        early_stopping_rounds=50,
        random_state=42,
    )
    model.fit(train[xgb_feats], train[target_col],
              eval_set=[(val[xgb_feats], val[target_col])], verbose=False)
    return model

m = fit_xgb(max_depth=2)
xgb_val_probs = m.predict_proba(val[xgb_feats])[:, 1]

print("Best number of trees:", m.best_iteration + 1)
print("XGBoost (depth 2) on validation:",
      f"acc {accuracy_score(val[target_col], (xgb_val_probs >= 0.5).astype(int)):.4f}",
      f"log loss {log_loss(val[target_col], xgb_val_probs):.4f}")

Best number of trees: 136
XGBoost (depth 2) on validation: acc 0.6545 log loss 0.6339


In [5]:
results = []
for depth in (1, 2, 3, 4):
    m = fit_xgb(max_depth=depth)
    probs = m.predict_proba(val[xgb_feats])[:, 1]
    ll = log_loss(val[target_col], probs)
    results.append((depth, ll, m.best_iteration + 1))
    print(f"depth {depth}: val log loss {ll:.4f}, trees {m.best_iteration + 1}")

best_depth, best_ll, best_trees = min(results, key=lambda r: r[1])
print(f"\nChosen: depth {best_depth} (val log loss {best_ll:.4f})")

depth 1: val log loss 0.6325, trees 222
depth 2: val log loss 0.6339, trees 136
depth 3: val log loss 0.6357, trees 101
depth 4: val log loss 0.6373, trees 99

Chosen: depth 1 (val log loss 0.6325)


In [6]:
trainval = pd.concat([train, val])

final_lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
final_lr.fit(trainval[lr_feats], trainval[target_col])
lr_test_probs = final_lr.predict_proba(test[lr_feats])[:, 1]

n_trees = int(best_trees * 1.1) + 1
final_xgb = XGBClassifier(
    n_estimators=n_trees, learning_rate=0.02, max_depth=best_depth,
    min_child_weight=20, subsample=0.8, colsample_bytree=0.8,
    reg_lambda=5, eval_metric="logloss", random_state=42,
)
final_xgb.fit(trainval[xgb_feats], trainval[target_col])
xgb_test_probs = final_xgb.predict_proba(test[xgb_feats])[:, 1]

y_test = test[target_col]
for name, p in (("Logistic (Old 8 + Elo)", lr_test_probs), ("XGBoost", xgb_test_probs)):
    print(f"{name:<24} acc {accuracy_score(y_test, (p >= 0.5).astype(int)):.4f}   log loss {log_loss(y_test, p):.4f}")

Logistic (Old 8 + Elo)   acc 0.6475   log loss 0.6193
XGBoost                  acc 0.6486   log loss 0.6243
